# My SmartQ Model Training & Evaluation

I use this notebook to reproduce the official model comparison in my SmartQ proposal.

I train:

- Linear Regression
- Random Forest
- XGBoost

I evaluate them mainly with:

- MAE
- RMSE

I fixed my official selection rule before final test evaluation:

> I select the model with the lowest validation MAE.


## 1. My validation result summary

My official validation results are:

| Model | Validation MAE | Validation RMSE |
|---|---:|---:|
| Linear Regression | 4.1146 | 6.1452 |
| Random Forest | 2.6315 | 4.4920 |
| **XGBoost** | **2.6302** | **4.3893** |

XGBoost has the lowest validation MAE, so I select it according to my declared rule.

Random Forest is extremely close.

I therefore describe the result as a near-tie rather than pretending XGBoost is dramatically better.


## 2. My final unseen test results

After I fixed the model-selection decision using validation data, I evaluated the later test period.

| Model | Test MAE | Test RMSE |
|---|---:|---:|
| Linear Regression | 4.0645 | 6.4606 |
| Random Forest | 2.5231 | 4.6425 |
| **Selected XGBoost** | **2.5824** | **4.9561** |
| Mean baseline | 14.9850 | 24.8143 |
| SmartQ deterministic ETA | 4.6386 | 6.3683 |

Random Forest happens to have a slightly lower test MAE.

I do **not** switch models after seeing that result because the test set is not supposed to select the model.

I keep XGBoost because it won using the validation rule I defined in advance.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Run from repository root or notebooks/.
DATASET_NAME = 'SmartQ_Synthetic_Operational_Dataset_100k.csv'
candidate_paths = [Path('data') / DATASET_NAME, Path('..') / 'data' / DATASET_NAME]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('SmartQ dataset not found.')

df = pd.read_csv(data_path, parse_dates=['scenario_date'])
completed = df[df['status'] == 'COMPLETED'].copy()


In [ ]:
TARGET = 'actual_wait_minutes'
numeric_features = [
    'arrival_offset_minutes','people_ahead','general_waiting','priority_waiting',
    'serving_count','open_general_counters','open_priority_counters','effective_open_counters',
    'counter_utilisation','queue_pressure_index','workload_minutes_ahead',
    'recent_avg_service_minutes_10','recent_avg_wait_minutes_10','recent_throughput_60m',
    'service_target_minutes','hour_of_day'
]
categorical_features = ['branch_code','service_code','booking_source','queue_type','day_of_week','is_peak_period']
features = numeric_features + categorical_features

dates = sorted(completed['scenario_date'].dt.normalize().unique())
n = len(dates)
train_end = pd.Timestamp(dates[int(n*0.70)-1])
val_start = pd.Timestamp(dates[int(n*0.70)])
val_end = pd.Timestamp(dates[int(n*0.85)-1])
test_start = pd.Timestamp(dates[int(n*0.85)])

train = completed[completed['scenario_date'] <= train_end].copy()
validation = completed[(completed['scenario_date'] >= val_start) & (completed['scenario_date'] <= val_end)].copy()
test = completed[completed['scenario_date'] >= test_start].copy()

X_train,y_train = train[features],train[TARGET]
X_val,y_val = validation[features],validation[TARGET]
X_test,y_test = test[features],test[TARGET]

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features),
])
Xtr = preprocessor.fit_transform(X_train)
Xv = preprocessor.transform(X_val)
Xt = preprocessor.transform(X_test)


## 3. I use baselines

My project requires a simple mean-wait baseline.

I also report the existing deterministic SmartQ ETA as an extra engineering comparison.

I do not use the deterministic ETA as an official model input because I want the ML models to learn directly from the queue conditions.


In [ ]:
def score(y_true, y_pred):
    y_pred = np.clip(np.asarray(y_pred), 0, None)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
    }

mean_wait = y_train.mean()
print('Mean baseline:', score(y_val, np.full(len(y_val), mean_wait)))
print('Deterministic ETA:', score(y_val, validation['baseline_eta_minutes']))


## 4. I train Linear Regression

I use Linear Regression as my simple, interpretable reference model.

I want to see how far a straightforward linear relationship can go before I move to more flexible tree models.


In [ ]:
linear = LinearRegression()
linear.fit(Xtr, y_train)
linear_val_pred = np.clip(linear.predict(Xv), 0, None)
print(score(y_val, linear_val_pred))


## 5. I train and lightly tune Random Forest

I compare two deliberately small parameter combinations.

I keep tuning limited because I want the experiment to stay understandable and inside the project scope.

I do not want a huge parameter search to overfit my validation data or turn the project into a tuning competition.


In [ ]:
rf_candidates = [
    {'n_estimators':100,'max_depth':18,'min_samples_leaf':2,'max_features':0.8},
    {'n_estimators':150,'max_depth':14,'min_samples_leaf':2,'max_features':1.0},
]

rf_trials = []
for params in rf_candidates:
    model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)
    model.fit(Xtr, y_train)
    pred = np.clip(model.predict(Xv), 0, None)
    rf_trials.append({**params, **score(y_val, pred)})
display(pd.DataFrame(rf_trials))


## 6. I train and lightly tune XGBoost

I compare a small number of controlled XGBoost configurations.

I use the same training and validation data as the other models so the comparison stays fair.


In [ ]:
xgb_candidates = [
    {'n_estimators':150,'max_depth':4,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9,'reg_lambda':1.0},
    {'n_estimators':200,'max_depth':5,'learning_rate':0.08,'subsample':0.9,'colsample_bytree':0.9,'reg_lambda':2.0},
]

xgb_trials = []
for params in xgb_candidates:
    model = XGBRegressor(random_state=42, n_jobs=-1, objective='reg:squarederror', tree_method='hist', **params)
    model.fit(Xtr, y_train)
    pred = np.clip(model.predict(Xv), 0, None)
    xgb_trials.append({**params, **score(y_val, pred)})
display(pd.DataFrame(xgb_trials))


## 7. I select XGBoost and check traffic scenarios

I select XGBoost using validation MAE.

On my final synthetic test period:

- MAE = **2.5824 minutes**
- RMSE = **4.9561 minutes**

For reporting, I group traffic using `queue_pressure_index`:

- Low: < 1.0
- Moderate: 1.0 to < 2.5
- Busy: >= 2.5

My test MAE increases from about **1.22 minutes in Low traffic** to **7.94 minutes in Busy traffic**.

I treat that as an important limitation rather than hiding it.


## 8. My modelling conclusion

I trained all three required models on the same chronologically separated data and evaluated them with the same MAE/RMSE metrics.

XGBoost achieved the lowest validation MAE, so I keep it as my SmartQ integration candidate.

I do not use the final test set to re-select the winner.

That keeps my evaluation method consistent with the rule I defined before seeing the final answers.
